In [ ]:
'''
과거 평균 혼잡도 데이터 만들기
160명 100%

- get-off 데이터와 car 데이터를 각각 DataFrame으로 변환
- 요일, 시각(10분 간격), 역코드, 상하행선 기준으로 그룹화하여 평균값 계산
- 결과를 CSV 파일로 저장

열차혼잡도
= 해당역의 혼잡도

승강장혼잡도
탑승정도 = 다음역 혼잡도 - 해당역 혼잡도(1-(하차비율/100))
탑승인원 = 탑승정도 * (160/100)
승강장 혼잡도 = (탑승인원/면적) / 4.3 *100

종합혼잡도
= 다음역 혼잡도
'''



import json
import pandas as pd
import numpy as np
import ast

PLATFORM_AREA = 6.0  # 승강장 면적 (m^2)
CAR_CAPACITY = 160   # 열차 1량 정원 (명)
STD_DENSITY = 4.3    # 기준 밀도 (명/m^2)

# === 1. 데이터 로드 ===
print("[1] CSV 데이터 로드 중...")

try:
    base_path = 'C:/Users/SSAFY/Desktop/juhi/S14P11A204/Congestion_Algorithm/data/'
    car_data = pd.read_csv(base_path + 'tmap_puzzle_filtering_car.csv', encoding='utf-8')
    get_off_data = pd.read_csv(base_path + 'tmap_puzzle_filtering_get-off.csv', encoding='utf-8')
    print(f" -> 완료 (Car: {car_data.shape}, Get-off: {get_off_data.shape})")
except FileNotFoundError:
    print(" -> [Error] 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()
except Exception as e:
    print(f" -> [Error] 데이터 로드 중 오류 발생: {e}")
    exit()


# === 2. 데이터 타입 변환 (String -> List) ===
print("[2] 데이터 타입 변환 중...")

def safe_literal_eval(val):
    """문자열 형태의 리스트('[10, 20]')를 실제 리스트 객체로 변환"""
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return val

# 타겟 컬럼 변환 적용
if 'congestionCar' in car_data.columns:
    car_data['congestionCar'] = car_data['congestionCar'].apply(safe_literal_eval)

if 'getOffCarRate' in get_off_data.columns:
    get_off_data['getOffCarRate'] = get_off_data['getOffCarRate'].apply(safe_literal_eval)


# === 3. 그룹화 및 평균 계산 함수 ===
def calculate_average_by_group(df, value_col):
    """요일, 시각, 역, 방향 기준으로 그룹화하여 리스트 요소별 평균 계산"""
    group_cols = [
            'subwayLine',
            'stationName', 
            'stationCode', 
            'updnLine', 
            'dow', 
            'hh', 
            'mm',
            'prevStationName',
            'prevStationCode'
        ]
    existing_cols = [c for c in group_cols if c in df.columns]

    if not existing_cols:
        print(f"[Error] 그룹화할 컬럼 부족: {df.columns}")
        return pd.DataFrame()

    def list_mean(series):
        # Series 내의 리스트들을 2차원 배열로 변환 후 수직 평균 계산
        matrix = np.array(series.tolist())
        if matrix.ndim == 1: 
            return matrix.tolist()
        return np.mean(matrix, axis=0).tolist()

    return df.groupby(existing_cols)[value_col].apply(list_mean).reset_index()


# === 4. 평균 계산 및 병합 실행 ===
print("[3] 그룹별 평균 계산 및 병합 중...")

if 'congestionCar' in car_data.columns:
    df_car_avg = calculate_average_by_group(car_data, 'congestionCar')
else:
    df_car_avg = pd.DataFrame()

if 'getOffCarRate' in get_off_data.columns:
    df_getoff_avg = calculate_average_by_group(get_off_data, 'getOffCarRate')
else:
    df_getoff_avg = pd.DataFrame()

# [핵심 수정] 병합 시 'prevStationCode'를 제외하고 공통 컬럼만 사용
# 이렇게 하면 car_data에만 있는 'prevStationCode'는 병합 결과에 그대로 보존됩니다.
merge_keys = [
    'subwayLine',
    'stationName', 
    'stationCode', 
    'updnLine', 
    'dow', 
    'hh', 
    'mm',
    'prevStationName',
    'prevStationCode'
]

df_merged = pd.merge(
    df_car_avg, 
    df_getoff_avg, 
    on=merge_keys, 
    how='inner'
)

# === [추가] 5. 승강장 혼잡도 계산 로직 ===
print("[4] 다음 역 혼잡도 매칭 중 (Self-Join)...")

# 5-1. 다음 역 혼잡도 가져오기 (Shift Logic)
# 정확한 계산을 위해 정렬: 요일 > 방향 > 시간 > 역 순서
# (주의: stationCode가 노선 순서대로 되어있다고 가정합니다.)
df_next_lookup = df_merged[['dow', 'hh', 'mm', 'updnLine', 'prevStationCode', 'congestionCar']].copy()
df_next_lookup.rename(columns={'congestionCar': 'next_station_congestion'}, inplace=True)

df_final = pd.merge(
    df_merged,
    df_next_lookup,
    left_on=['dow', 'hh', 'mm', 'updnLine', 'stationCode'],   # 나의 역 코드
    right_on=['dow', 'hh', 'mm', 'updnLine', 'prevStationCode'], # 다음 역이 가리키는 이전 역 코드
    how='left', # 다음 역이 없는 경우(종점)도 데이터 유지
    suffixes=('', '_next_dup') # 중복 컬럼 처리
)

# 불필요한 중복 컬럼 제거 (prevStationCode_next_dup 등)
df_final = df_final.loc[:, ~df_final.columns.str.endswith('_next_dup')]

# === 6. 승강장 혼잡도 계산 로직 ===
print("[5] 승강장 혼잡도 계산 중...")

# 6-1. 종합 혼잡도 (Total Congestion) = 다음 역 혼잡도
df_final['totalCongestion'] = df_final['next_station_congestion'].apply(
    lambda x: x if isinstance(x, list) else []
)
# 6-2. 승강장 혼잡도 계산 함수
def calc_platform_congestion(row):
    # 다음 역 데이터가 없으면(NaN) 빈 리스트 (종점인 경우)
    if isinstance(row['next_station_congestion'], float): # NaN check
        return []
    if not isinstance(row['next_station_congestion'], list) or not row['congestionCar']:
        return []

    current_cong = np.array(row['congestionCar'])
    next_cong = np.array(row['next_station_congestion'])
    get_off_rate = np.array(row['getOffCarRate'])

    if len(current_cong) != len(next_cong):
        return []

    # [공식 적용]
    remaining_ratio = current_cong * (1 - (get_off_rate / 100.0))
    boarding_level = next_cong - remaining_ratio
    boarding_level = np.maximum(boarding_level, 0) # 음수 방지

    boarding_people = boarding_level * (CAR_CAPACITY / 100.0)
    density = boarding_people / PLATFORM_AREA
    platform_congestion = (density / STD_DENSITY) * 100

    return np.round(platform_congestion, 2).tolist()

# 5-3. 함수 적용
df_final['platformCongestion'] = df_final.apply(calc_platform_congestion, axis=1)

print(df_final[['stationCode', 'congestionCar', 'platformCongestion', 'totalCongestion']].head())

In [ ]:
df_final